# Exploration de l'API Open-Meteo
Objectif : récupérer la météo horaire de chaque parc français (température, pluie, type de temps, vent)
pour la croiser avec les temps d'attente collectés via Queue-Times.

In [7]:
import requests
import pandas as pd

# Liste des parcs français et de leurs coordonnées GPS (endpoint parks.json de Queue-Times)
reponse = requests.get("https://queue-times.com/parks.json", timeout=10)
print("Code de statut :", reponse.status_code)

lignes_parcs = []
for groupe in reponse.json():
    for parc in groupe["parks"]:
        lignes_parcs.append({
            "id_parc": parc["id"],
            "nom_parc": parc["name"],
            "pays": parc["country"],
            "latitude": parc["latitude"],
            "longitude": parc["longitude"],
        })

df_parcs = pd.DataFrame(lignes_parcs)
parcs_france = df_parcs[df_parcs["pays"] == "France"].reset_index(drop=True)
print("Nombre de parcs en France :", len(parcs_france))
parcs_france



Code de statut : 200
Nombre de parcs en France : 10


,id_parc,nom_parc,pays,latitude,longitude
0,291,Futuroscope,France,46.669167,0.368889
1,9,Parc Astérix,France,49.1341839,2.5712301
2,301,Walibi Rhône-Alpes,France,45.621389,5.570278
3,303,Le Pal,France,46.507222,3.630556
4,327,Le Parc du Petit Prince,France,47.860539,7.297752
5,336,Nigloland,France,48.2633,4.607
6,320,PanoraMagique,France,48.86901442,2.78646006
7,304,Vulcania,France,45.813449,2.940501
8,28,Disney Adventure World Paris,France,48.8673,2.779008
9,4,Disneyland Park Paris,France,48.8722344,2.7758079


In [8]:
# Coordonnées GPS : stockées en texte dans parks.json, converties en nombres
parcs_france["latitude"] = parcs_france["latitude"].astype(float)
parcs_france["longitude"] = parcs_france["longitude"].astype(float)
print(parcs_france[["id_parc", "nom_parc", "latitude", "longitude"]].dtypes)

# Test sur un seul parc : Disneyland Park Paris (id 4)
disney = parcs_france[parcs_france["id_parc"] == 4].iloc[0]

url_meteo = "https://api.open-meteo.com/v1/forecast"
params = {
    "latitude": disney["latitude"],
    "longitude": disney["longitude"],
    "hourly": "temperature_2m,precipitation,weather_code,wind_speed_10m",
    "timezone": "UTC",
    "past_days": 2,
    "forecast_days": 1,
}
reponse = requests.get(url_meteo, params=params, timeout=10)
print("\nCode de statut :", reponse.status_code)

meteo = reponse.json()
print("Clés du JSON :", meteo.keys())
print("Unités :", meteo["hourly_units"])
print("Nombre d'heures reçues :", len(meteo["hourly"]["time"]))

df_test = pd.DataFrame(meteo["hourly"])
df_test.head()

id_parc        int64
nom_parc         str
latitude     float64
longitude    float64
dtype: object

Code de statut : 200
Clés du JSON : dict_keys(['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'hourly_units', 'hourly'])
Unités : {'time': 'iso8601', 'temperature_2m': '°C', 'precipitation': 'mm', 'weather_code': 'wmo code', 'wind_speed_10m': 'km/h'}
Nombre d'heures reçues : 72


,time,temperature_2m,precipitation,weather_code,wind_speed_10m
0,2026-10-11T00:00,12.2,0.0,3,12.4
1,2026-10-11T01:00,11.6,0.0,3,10.4
2,2026-10-11T02:00,11.2,0.0,3,10.2
3,2026-10-11T03:00,11.6,0.1,61,10.8
4,2026-10-11T04:00,11.3,0.0,3,11.3


## Collecte de la météo pour les 10 parcs
On garde uniquement les heures déjà passées : les heures à venir ne sont que des prévisions.

In [9]:
from datetime import datetime, timezone

def collecter_meteo(id_parc, nom_parc, latitude, longitude):
    """Récupère la météo horaire d'un parc et renvoie un DataFrame."""
    params = {
        "latitude": latitude,
        "longitude": longitude,
        "hourly": "temperature_2m,precipitation,weather_code,wind_speed_10m",
        "timezone": "UTC",
        "past_days": 2,
        "forecast_days": 1,
    }
    try:
        reponse = requests.get(url_meteo, params=params, timeout=10)
        reponse.raise_for_status()
    except requests.exceptions.RequestException as erreur:
        print(f"Erreur pour {nom_parc} : {erreur}")
        return pd.DataFrame()

    df = pd.DataFrame(reponse.json()["hourly"])
    df["id_parc"] = id_parc
    df["nom_parc"] = nom_parc
    print(f"{nom_parc} : {len(df)} heures récupérées")
    return df


# Boucle sur les 10 parcs
tables = []
for _, parc in parcs_france.iterrows():
    tables.append(collecter_meteo(parc["id_parc"], parc["nom_parc"], parc["latitude"], parc["longitude"]))

df_meteo = pd.concat(tables, ignore_index=True)

# Renommage en français et conversion de l'heure en vraie date UTC
df_meteo = df_meteo.rename(columns={
    "time": "heure_utc",
    "temperature_2m": "temperature_c",
    "precipitation": "precipitation_mm",
    "weather_code": "code_meteo",
    "wind_speed_10m": "vent_kmh",
})
df_meteo["heure_utc"] = pd.to_datetime(df_meteo["heure_utc"], utc=True)

# On ne garde que les heures passées
maintenant = datetime.now(timezone.utc)
nb_avant = len(df_meteo)
df_meteo = df_meteo[df_meteo["heure_utc"] < maintenant]
print(f"\nLignes avant filtre : {nb_avant} | Heures futures retirées : {nb_avant - len(df_meteo)} | Lignes gardées : {len(df_meteo)}")

df_meteo.head()

Futuroscope : 72 heures récupérées
Parc Astérix : 72 heures récupérées
Walibi Rhône-Alpes : 72 heures récupérées
Le Pal : 72 heures récupérées
Le Parc du Petit Prince : 72 heures récupérées
Nigloland : 72 heures récupérées
PanoraMagique : 72 heures récupérées
Vulcania : 72 heures récupérées
Disney Adventure World Paris : 72 heures récupérées
Disneyland Park Paris : 72 heures récupérées

Lignes avant filtre : 720 | Heures futures retirées : 140 | Lignes gardées : 580


,heure_utc,temperature_c,precipitation_mm,code_meteo,vent_kmh,id_parc,nom_parc
0,2026-10-11 00:00:00+00:00,8.5,0.0,3,3.2,291,Futuroscope
1,2026-10-11 01:00:00+00:00,8.0,0.0,3,2.8,291,Futuroscope
2,2026-10-11 02:00:00+00:00,7.6,0.0,3,3.6,291,Futuroscope
3,2026-10-11 03:00:00+00:00,6.3,0.0,3,3.1,291,Futuroscope
4,2026-10-11 04:00:00+00:00,6.2,0.0,3,3.7,291,Futuroscope


## Exploration de la météo

In [10]:
# 1. Structure et types
print("=== Types des colonnes ===")
df_meteo.info()

# 2. Valeurs manquantes
print("\n=== Valeurs manquantes ===")
print(df_meteo.isna().sum())

# 3. Doublons (clé : un parc, une heure)
print("\n=== Doublons (id_parc + heure_utc) ===")
print(df_meteo.duplicated(subset=["id_parc", "heure_utc"]).sum())

# 4. Période couverte et nombre d'heures par parc
print("\n=== Période couverte ===")
print("Première heure :", df_meteo["heure_utc"].min())
print("Dernière heure :", df_meteo["heure_utc"].max())
print("\n=== Nombre d'heures par parc ===")
print(df_meteo["nom_parc"].value_counts())

# 5. Statistiques des mesures, par parc
print("\n=== Température (°C) par parc ===")
print(df_meteo.groupby("nom_parc")["temperature_c"].describe().round(1))

print("\n=== Pluie (mm) par parc : total et nombre d'heures pluvieuses ===")
print(df_meteo.groupby("nom_parc")["precipitation_mm"].agg(
    total_mm="sum",
    heures_avec_pluie=lambda x: (x > 0).sum(),
).round(1))

print("\n=== Vent (km/h) ===")
print(df_meteo["vent_kmh"].describe().round(1))

# 6. Types de temps (codes WMO) rencontrés
print("\n=== Codes météo rencontrés ===")
print(df_meteo["code_meteo"].value_counts().sort_index())

=== Types des colonnes ===
<class 'pandas.DataFrame'>
Index: 580 entries, 0 to 705
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype              
---  ------            --------------  -----              
 0   heure_utc         580 non-null    datetime64[us, UTC]
 1   temperature_c     580 non-null    float64            
 2   precipitation_mm  580 non-null    float64            
 3   code_meteo        580 non-null    int64              
 4   vent_kmh          580 non-null    float64            
 5   id_parc           580 non-null    int64              
 6   nom_parc          580 non-null    str                
dtypes: datetime64[us, UTC](1), float64(3), int64(2), str(1)
memory usage: 44.9 KB

=== Valeurs manquantes ===
heure_utc           0
temperature_c       0
precipitation_mm    0
code_meteo          0
vent_kmh            0
id_parc             0
nom_parc            0
dtype: int64

=== Doublons (id_parc + heure_utc) ===
0

=== Période couverte ===
Première

## Nettoyage et enrichissement de la météo

In [12]:
# Traduction des codes météo WMO en libellés (table officielle WMO, regroupée)
def libelle_meteo(code):
    if code == 0:
        return "Ciel dégagé"
    if code in (1, 2):
        return "Peu nuageux"
    if code == 3:
        return "Couvert"
    if code in (45, 48):
        return "Brouillard"
    if 51 <= code <= 57:
        return "Bruine"
    if 61 <= code <= 67 or 80 <= code <= 82:
        return "Pluie"
    if 71 <= code <= 77 or code in (85, 86):
        return "Neige"
    if 95 <= code <= 99:
        return "Orage"
    return "Autre"

df_meteo_clean = df_meteo.copy().reset_index(drop=True)
df_meteo_clean["libelle_meteo"] = df_meteo_clean["code_meteo"].apply(libelle_meteo)
df_meteo_clean["date_collecte"] = datetime.now(timezone.utc)

# Vérifications
print("Index : de", df_meteo_clean.index.min(), "à", df_meteo_clean.index.max(), "pour", len(df_meteo_clean), "lignes")
print("\n=== Correspondance code → libellé ===")
print(df_meteo_clean.groupby(["code_meteo", "libelle_meteo"]).size())
print("\nLignes classées « Autre » :", (df_meteo_clean["libelle_meteo"] == "Autre").sum())
df_meteo_clean.head()

Index : de 0 à 579 pour 580 lignes

=== Correspondance code → libellé ===
code_meteo  libelle_meteo
0           Ciel dégagé       54
1           Peu nuageux       76
2           Peu nuageux       58
3           Couvert          358
45          Brouillard        22
61          Pluie             12
dtype: int64

Lignes classées « Autre » : 0


,heure_utc,temperature_c,precipitation_mm,code_meteo,vent_kmh,id_parc,nom_parc,libelle_meteo,date_collecte
0,2026-10-11 00:00:00+00:00,8.5,0.0,3,3.2,291,Futuroscope,Couvert,2026-10-13 11:55:23.862919+00:00
1,2026-10-11 01:00:00+00:00,8.0,0.0,3,2.8,291,Futuroscope,Couvert,2026-10-13 11:55:23.862919+00:00
2,2026-10-11 02:00:00+00:00,7.6,0.0,3,3.6,291,Futuroscope,Couvert,2026-10-13 11:55:23.862919+00:00
3,2026-10-11 03:00:00+00:00,6.3,0.0,3,3.1,291,Futuroscope,Couvert,2026-10-13 11:55:23.862919+00:00
4,2026-10-11 04:00:00+00:00,6.2,0.0,3,3.7,291,Futuroscope,Couvert,2026-10-13 11:55:23.862919+00:00


Avant de conseiller un client selon la météo, on vérifie sur quelles conditions météo on a des données ; cette semaine, on peut comparer temps dégagé et temps couvert, mais pas encore l’effet de la pluie.